# 🌳 Capítulo 8 — Árboles (Trees)
## Notebook 2: Recorridos de Árboles y Patrón Template Method

**Libro:** Goodrich, Tamassia & Goldwasser — *Data Structures and Algorithms in Python*, §8.4  
**Materia:** Estructuras de Datos y Algoritmos — Lic. en Sistemas | **Año:** 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap08/02_Recorridos_Arboles_Teoria.ipynb)

## 🎯 Objetivos de aprendizaje

1. Implementar los cuatro algoritmos de recorrido: **preorden, postorden, anchura (BFS), inorden**.
2. Analizar por qué todos son O(n).
3. Implementar la clase `EulerTour` usando el **Patrón Template Method**.
4. Especializar `EulerTour` para resolver problemas concretos (tabla de contenidos, espacio en disco).
5. Construir y evaluar un `ExpressionTree` usando `BinaryEulerTour`.

## 📖 Contenidos

| Sección | Tema |
|---------|------|
| §8.4.1 | Preorden (Preorder) |
| §8.4.2 | Postorden (Postorder) |
| §8.4.3 | Anchura / BFS (Breadth-First) |
| §8.4.4 | Inorden (Inorder) — solo árboles binarios |
| §8.4.5 | Recorrido de Euler y patrón Template Method |
| §8.4.6 | Caso de estudio: ExpressionTree |

## 🔧 Configuración del entorno

In [ ]:
import sys
from abc import ABCMeta, abstractmethod
from collections import deque

print(f"Python {sys.version}")

# ── Reutilización del ADT Tree y SimpleTree del NB01 ─────────────────────────
# (En un entorno real, estas clases estarían en un módulo compartido)

class Tree(metaclass=ABCMeta):
    class Position(metaclass=ABCMeta):
        @abstractmethod
        def element(self): raise NotImplementedError
        @abstractmethod
        def __eq__(self, other): raise NotImplementedError
        def __ne__(self, other): return not (self == other)

    @abstractmethod
    def root(self): raise NotImplementedError
    @abstractmethod
    def parent(self, p): raise NotImplementedError
    @abstractmethod
    def num_children(self, p): raise NotImplementedError
    @abstractmethod
    def children(self, p): raise NotImplementedError
    @abstractmethod
    def __len__(self): raise NotImplementedError

    def is_root(self, p): return self.root() == p
    def is_leaf(self, p): return self.num_children(p) == 0
    def is_empty(self): return len(self) == 0

    def depth(self, p):
        return 0 if self.is_root(p) else 1 + self.depth(self.parent(p))

    def height(self, p=None):
        if p is None: p = self.root()
        return 0 if self.is_leaf(p) else 1 + max(self.height(c) for c in self.children(p))

    def positions(self):
        return self.breadthfirst()

    def breadthfirst(self):
        if not self.is_empty():
            q = deque([self.root()])
            while q:
                p = q.popleft()
                yield p
                for c in self.children(p): q.append(c)

    def __iter__(self):
        for p in self.positions(): yield p.element()


class BinaryTree(Tree):
    @abstractmethod
    def left(self, p): raise NotImplementedError
    @abstractmethod
    def right(self, p): raise NotImplementedError

    def sibling(self, p):
        parent = self.parent(p)
        if parent is None: return None
        if p == self.left(parent): return self.right(parent)
        else: return self.left(parent)

    def children(self, p):
        if self.left(p) is not None: yield self.left(p)
        if self.right(p) is not None: yield self.right(p)

    def inorder(self):
        if not self.is_empty():
            for p in self._subtree_inorder(self.root()):
                yield p

    def _subtree_inorder(self, p):
        if self.left(p) is not None:
            for other in self._subtree_inorder(self.left(p)):
                yield other
        yield p
        if self.right(p) is not None:
            for other in self._subtree_inorder(self.right(p)):
                yield other

    def positions(self):
        return self.inorder()


class LinkedBinaryTree(BinaryTree):
    class _Node:
        __slots__ = '_element','_parent','_left','_right'
        def __init__(self, e, parent=None, left=None, right=None):
            self._element, self._parent = e, parent
            self._left, self._right = left, right

    class _Position(BinaryTree.Position):
        def __init__(self, container, node):
            self._container, self._node = container, node
        def element(self): return self._node._element
        def __eq__(self, other):
            return type(other) is type(self) and other._node is self._node

    def _validate(self, p):
        if not isinstance(p, self._Position): raise TypeError
        if p._container is not self: raise ValueError
        if p._node._parent is p._node: raise ValueError('p is no longer valid')
        return p._node

    def _make_position(self, node):
        return self._Position(self, node) if node is not None else None

    def __init__(self):
        self._root = None
        self._size = 0

    def __len__(self): return self._size
    def root(self): return self._make_position(self._root)
    def parent(self, p): return self._make_position(self._validate(p)._parent)
    def left(self, p): return self._make_position(self._validate(p)._left)
    def right(self, p): return self._make_position(self._validate(p)._right)
    def num_children(self, p):
        node = self._validate(p)
        return (node._left is not None) + (node._right is not None)

    def add_root(self, e):
        if self._root is not None: raise ValueError('Root exists')
        self._size = 1
        self._root = self._Node(e)
        return self._make_position(self._root)

    def add_left(self, p, e):
        node = self._validate(p)
        if node._left is not None: raise ValueError('Left child exists')
        self._size += 1
        node._left = self._Node(e, parent=node)
        return self._make_position(node._left)

    def add_right(self, p, e):
        node = self._validate(p)
        if node._right is not None: raise ValueError('Right child exists')
        self._size += 1
        node._right = self._Node(e, parent=node)
        return self._make_position(node._right)

print("Clases Tree, BinaryTree, LinkedBinaryTree cargadas ✓")

---

## 🔁 §8.4.1 Recorrido en Preorden (Preorder)

**Algoritmo:** visitar la raíz, luego recursivamente cada subárbol.

```
preorder(p):
    visitar(p)
    for each child c of p:
        preorder(c)
```

**Aplicación típica:** tabla de contenidos (el capítulo antes que sus secciones)  
**Complejidad:** O(n) — cada nodo visitado exactamente una vez.

In [ ]:
def preorder(T, p=None, indent=0):
    """Recorrido preorden recursivo — retorna lista de (nivel, elemento)."""
    if p is None: p = T.root()
    result = [(indent, p.element())]
    for c in T.children(p):
        result.extend(preorder(T, c, indent + 1))
    return result

# Árbol de ejemplo: libro
from collections import namedtuple
class SimpleTree(Tree):
    class _Node:
        __slots__ = '_e','_parent','_children'
        def __init__(self, e, parent=None):
            self._e, self._parent, self._children = e, parent, []
    class _Position(Tree.Position):
        def __init__(self, c, n): self._container, self._node = c, n
        def element(self): return self._node._e
        def __eq__(self, other): return type(other) is type(self) and other._node is self._node
    def _validate(self, p):
        if not isinstance(p, self._Position): raise TypeError
        if p._container is not self: raise ValueError
        return p._node
    def _make_position(self, node): return self._Position(self, node) if node else None
    def __init__(self): self._root = None; self._size = 0
    def __len__(self): return self._size
    def root(self): return self._make_position(self._root)
    def parent(self, p): return self._make_position(self._validate(p)._parent)
    def num_children(self, p): return len(self._validate(p)._children)
    def children(self, p):
        for n in self._validate(p)._children: yield self._make_position(n)
    def add_root(self, e):
        self._root = self._Node(e); self._size = 1; return self._make_position(self._root)
    def add_child(self, p, e):
        n = self._validate(p); child = self._Node(e, parent=n)
        n._children.append(child); self._size += 1; return self._make_position(child)

# Construir árbol libro
T_book = SimpleTree()
r = T_book.add_root("Libro DS&A")
cap1 = T_book.add_child(r, "Cap. 1 Python")
cap2 = T_book.add_child(r, "Cap. 2 OOP")
cap3 = T_book.add_child(r, "Cap. 3 Análisis")
s1_1 = T_book.add_child(cap1, "§1.1 Tipos")
s1_2 = T_book.add_child(cap1, "§1.2 Funciones")
s2_1 = T_book.add_child(cap2, "§2.1 Clases")
s2_2 = T_book.add_child(cap2, "§2.2 Herencia")
s3_1 = T_book.add_child(cap3, "§3.1 Big-O")

print("=== Tabla de Contenidos (Preorden) ===")
for nivel, elem in preorder(T_book):
    print("  " * nivel + str(elem))

---

## 🔁 §8.4.2 Recorrido en Postorden (Postorder)

**Algoritmo:** recursivamente los subárboles, luego la raíz.

```
postorder(p):
    for each child c of p:
        postorder(c)
    visitar(p)
```

**Aplicación típica:** calcular espacio en disco (necesito saber el tamaño de los hijos antes que el del directorio).

In [ ]:
def postorder_disk_usage(T, p=None):
    """Calcula espacio en disco (postorden). Hojas tienen tamaño = 1KB por defecto."""
    if p is None: p = T.root()
    if T.is_leaf(p):
        return 1   # unidades arbitrarias
    else:
        total = sum(postorder_disk_usage(T, c) for c in T.children(p))
        return total

# Árbol Sistema de archivos
T_fs = SimpleTree()
root_fs = T_fs.add_root("/")
home = T_fs.add_child(root_fs, "home")
usr  = T_fs.add_child(root_fs, "usr")
docs = T_fs.add_child(home, "docs")
mus  = T_fs.add_child(home, "music")
_    = T_fs.add_child(docs, "CV.pdf")
_    = T_fs.add_child(docs, "tesis.pdf")
_    = T_fs.add_child(mus, "song.mp3")
_    = T_fs.add_child(usr, "bin")

print("=== Espacio en disco (Postorden) ===")
def postorder_print(T, p=None, indent=0):
    if p is None: p = T.root()
    for c in T.children(p):
        postorder_print(T, c, indent+1)
    size = postorder_disk_usage(T, p)
    print("  " * indent + f"{p.element():15s}  {size} KB")

postorder_print(T_fs)

---

## 🔁 §8.4.3 Recorrido en Anchura (Breadth-First Search)

**Algoritmo:** visitar todos los nodos de nivel d antes que los de nivel d+1. Usa una **cola**.

```
breadth_first():
    Q = deque([root])
    while Q no vacía:
        p = Q.popleft()
        visitar(p)
        for each child c of p: Q.append(c)
```

**Aplicación típica:** juegos de tablero (explorar movimientos por niveles)  
**Complejidad:** O(n) — cada nodo entra y sale de la cola exactamente una vez.

In [ ]:
def breadth_first_levels(T):
    """BFS mostrando el nivel de cada nodo."""
    if T.is_empty(): return
    current_level = [T.root()]
    level_num = 0
    while current_level:
        elems = [p.element() for p in current_level]
        print(f"  Nivel {level_num}: {elems}")
        next_level = []
        for p in current_level:
            for c in T.children(p):
                next_level.append(c)
        current_level = next_level
        level_num += 1

print("=== Recorrido BFS por niveles ===")
breadth_first_levels(T_book)

---

## 🔁 §8.4.4 Recorrido en Inorden (Inorder) — solo Árboles Binarios

**Algoritmo:** subárbol izquierdo → raíz → subárbol derecho.

```
inorder(p):
    if left(p) is not None: inorder(left(p))
    visitar(p)
    if right(p) is not None: inorder(right(p))
```

**Aplicación típica:** árbol de búsqueda binaria — el inorden visita los nodos en **orden creciente**.

In [ ]:
# Árbol binario de expresión: (3 + 1) * (9 - 5)
T_expr = LinkedBinaryTree()
root_e = T_expr.add_root('*')
left_e = T_expr.add_left(root_e, '+')
right_e = T_expr.add_right(root_e, '-')
T_expr.add_left(left_e, '3')
T_expr.add_right(left_e, '1')
T_expr.add_left(right_e, '9')
T_expr.add_right(right_e, '5')

print("Árbol de expresión: (3 + 1) * (9 - 5)")
print()
print("=== Inorden: reconstruye la expresión ===")
inorden = [p.element() for p in T_expr.inorder()]
print("  " + " ".join(inorden))

print()
print("=== Preorden: notación prefija (operador antes) ===")
def preorder_bt(T, p=None):
    if p is None: p = T.root()
    yield p.element()
    for c in T.children(p):
        yield from preorder_bt(T, c)
print("  " + " ".join(preorder_bt(T_expr)))

print()
print("=== Postorden: notación postfija (operador después) ===")
def postorder_bt(T, p=None):
    if p is None: p = T.root()
    for c in T.children(p):
        yield from postorder_bt(T, c)
    yield p.element()
print("  " + " ".join(postorder_bt(T_expr)))

---

## 🏗️ §8.4.5 Recorrido de Euler y Patrón Template Method

El **Recorrido de Euler** es un marco general que visita cada nodo **múltiples veces**:
- **hook_previsit(p, d, path)** — al llegar al nodo (antes de sus hijos)
- **hook_postvisit(p, d, path)** — al salir del nodo (después de sus hijos)
- **hook_invisit(p, d, path)** — entre hijo izquierdo y derecho (solo árboles binarios)

El algoritmo es el "Template" (plantilla); las subclases añaden comportamiento específico en los hooks.

```
EulerTour.execute():          # plantilla fija, no se cambia
    tour(root, depth=0, path=[])

EulerTour._tour(p, d, path):
    hook_previsit(p, d, path)          # HOOK 1
    results = []
    for child in p.children():
        path.append(...)
        results.append( _tour(child, d+1, path) )
        path.pop()
        hook_invisit(...)               # HOOK 2 (solo BinaryEulerTour)
    return hook_postvisit(p, d, path, results)  # HOOK 3
```

In [ ]:
class EulerTour:
    """Marco abstracto de recorrido de Euler usando Template Method (§8.4.5)."""

    def __init__(self, tree):
        self._tree = tree

    def tree(self):
        return self._tree

    def execute(self):
        """Ejecuta el recorrido de Euler y devuelve el resultado final."""
        if len(self._tree) > 0:
            return self._tour(self._tree.root(), 0, [])

    def _tour(self, p, d, path):
        self._hook_previsit(p, d, path)
        results = []
        path.append(0)
        for c in self._tree.children(p):
            results.append(self._tour(c, d + 1, path))
            path[-1] += 1
        path.pop()
        answer = self._hook_postvisit(p, d, path, results)
        return answer

    def _hook_previsit(self, p, d, path):   pass   # por defecto: no-op
    def _hook_postvisit(self, p, d, path, results):  return None


class BinaryEulerTour(EulerTour):
    """Euler Tour especializado para árboles binarios (agrega hook_invisit)."""

    def _tour(self, p, d, path):
        self._hook_previsit(p, d, path)
        results = [None, None]
        # recorrer subárbol izquierdo
        if self._tree.left(p) is not None:
            path.append(0)
            results[0] = self._tour(self._tree.left(p), d+1, path)
            path.pop()
        # hook entre izquierdo y derecho
        self._hook_invisit(p, d, path)
        # recorrer subárbol derecho
        if self._tree.right(p) is not None:
            path.append(1)
            results[1] = self._tour(self._tree.right(p), d+1, path)
            path.pop()
        return self._hook_postvisit(p, d, path, results)

    def _hook_invisit(self, p, d, path):  pass

print("Clases EulerTour y BinaryEulerTour definidas ✓")

In [ ]:
# ── Aplicación 1: Tabla de contenidos indentada con EulerTour ────────────────

class IndentedTOC(EulerTour):
    """Genera tabla de contenidos con indentación (previsit)."""
    def _hook_previsit(self, p, d, path):
        label = '.'.join(str(x+1) for x in path)
        indent = '  ' * d
        print(f"{indent}{label} {p.element()}")

print("=== Tabla de Contenidos con EulerTour ===")
toc = IndentedTOC(T_book)
toc.execute()

print()
# ── Aplicación 2: Calcular espacio en disco con EulerTour ────────────────────
class DiskUsageTour(EulerTour):
    """Calcula uso de disco (postvisit acumulado)."""
    def _hook_previsit(self, p, d, path):
        # Asignamos tamaño base a hojas = 1
        pass

    def _hook_postvisit(self, p, d, path, results):
        if self._tree.is_leaf(p):
            size = 1
        else:
            size = 1 + sum(r for r in results if r is not None)
        indent = '  ' * d
        print(f"{indent}{p.element()}: {size} unidades")
        return size

print("=== Uso de disco con EulerTour ===")
disk = DiskUsageTour(T_fs)
total = disk.execute()
print(f"Total: {total} unidades")

In [ ]:
# ── Aplicación 3: ExpressionTree con BinaryEulerTour ─────────────────────────

class ExpressionTree(LinkedBinaryTree):
    """Árbol de expresión aritmética (§8.4.6)."""

    def __init__(self, token, left=None, right=None):
        super().__init__()
        if not isinstance(token, str):
            raise TypeError('Token must be a string')
        self._root = self._Node(token)
        self._size = 1
        if left is not None:
            if token not in '+-*x/':
                raise ValueError('token must be an operator')
            # Attach left subtree
            self._root._left = left._root
            left._root._parent = self._root
            self._size += left._size
        if right is not None:
            self._root._right = right._root
            right._root._parent = self._root
            self._size += right._size

    def evaluate(self):
        """Evalúa la expresión numérica (postorden)."""
        def _eval(p):
            if self.is_leaf(p):
                return float(p.element())
            op = p.element()
            left_val  = _eval(self.left(p))
            right_val = _eval(self.right(p))
            if op == '+': return left_val + right_val
            if op == '-': return left_val - right_val
            if op in ('*', 'x'): return left_val * right_val
            if op == '/': return left_val / right_val
            raise ValueError(f'Unknown operator: {op}')
        return _eval(self.root())

    def __str__(self):
        """Representación parentizada (inorden con EulerTour)."""
        pieces = []
        def _infix(p):
            if self.is_leaf(p):
                pieces.append(p.element())
            else:
                pieces.append('(')
                _infix(self.left(p))
                pieces.append(p.element())
                _infix(self.right(p))
                pieces.append(')')
        _infix(self.root())
        return ''.join(pieces)


# Construir (3 + 1) * (9 - 5)
e1 = ExpressionTree('3')
e2 = ExpressionTree('1')
e3 = ExpressionTree('9')
e4 = ExpressionTree('5')
sum_tree  = ExpressionTree('+', e1, e2)   # 3 + 1
diff_tree = ExpressionTree('-', e3, e4)   # 9 - 5
prod_tree = ExpressionTree('*', sum_tree, diff_tree)  # (3+1) * (9-5)

print(f"Expresión: {prod_tree}")
print(f"Valor:     {prod_tree.evaluate()}")
print(f"Esperado:  {(3+1)*(9-5)}")  # 16

# Otro ejemplo: ((3 + 1) * (9 - 5)) / 2
e5 = ExpressionTree('2')
div_tree = ExpressionTree('/', prod_tree, e5)
print(f"\nExpresión: {div_tree}")
print(f"Valor:     {div_tree.evaluate()}")

---

## 📊 Comparación de recorridos

| Recorrido | Orden de visita | Usa | Aplicaciones típicas |
|-----------|-----------------|-----|----------------------|
| **Preorden** | raíz → hijos | recursión | TOC, copiar árbol, serialización |
| **Postorden** | hijos → raíz | recursión | tamaño en disco, evaluar expresiones |
| **BFS / Anchura** | nivel a nivel | cola (deque) | juegos, rutas más cortas |
| **Inorden** | izq → raíz → der | recursión | BST en orden, expresiones infijas |
| **Euler Tour** | pre+post (+in para BT) | Template Method | aplicaciones genéricas personalizables |

Todos son **O(n)** — cada nodo es visitado un número constante de veces.

## 🧪 Tests

In [ ]:
import unittest

class TestRecorridos(unittest.TestCase):

    def setUp(self):
        """Árbol binario:
             A
            / \\
           B   C
          / \\
         D   E
        """
        self.T = LinkedBinaryTree()
        A = self.T.add_root('A')
        B = self.T.add_left(A,  'B')
        C = self.T.add_right(A, 'C')
        D = self.T.add_left(B,  'D')
        E = self.T.add_right(B, 'E')
        self.A, self.B, self.C, self.D, self.E = A, B, C, D, E

    def test_inorden(self):
        result = [p.element() for p in self.T.inorder()]
        self.assertEqual(result, ['D','B','E','A','C'])

    def test_preorden(self):
        result = list(preorder_bt(self.T))
        self.assertEqual(result, ['A','B','D','E','C'])

    def test_postorden(self):
        result = list(postorder_bt(self.T))
        self.assertEqual(result, ['D','E','B','C','A'])

    def test_expression_tree_eval(self):
        # (3 + 1) * (9 - 5) = 16
        t = ExpressionTree('*',
                ExpressionTree('+', ExpressionTree('3'), ExpressionTree('1')),
                ExpressionTree('-', ExpressionTree('9'), ExpressionTree('5')))
        self.assertAlmostEqual(t.evaluate(), 16.0)

    def test_expression_tree_str(self):
        t = ExpressionTree('+', ExpressionTree('3'), ExpressionTree('1'))
        self.assertEqual(str(t), '(3+1)')

    def test_euler_tour_toc(self):
        """EulerTour produce previsit antes que hijos."""
        visits = []
        class RecordTour(EulerTour):
            def _hook_previsit(self, p, d, path):
                visits.append(p.element())
        RecordTour(self.T).execute()
        self.assertEqual(visits[0], 'A')   # raíz primero

suite = unittest.TestLoader().loadTestsFromTestCase(TestRecorridos)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\n{'✅ Todos los tests pasaron' if result.wasSuccessful() else '❌ Hubo fallos'}")

---

## ✅ Autoevaluación

- [ ] Simular a mano preorden, postorden e inorden sobre un árbol de 7 nodos
- [ ] Escribir `BFS` desde cero usando `deque` sin consultar el código
- [ ] Explicar la diferencia entre `hook_previsit` y `hook_postvisit`
- [ ] Crear una subclase de `EulerTour` que cuente nodos por nivel
- [ ] Construir el `ExpressionTree` de `(a + b) * (c - d)` y evaluar para a=2,b=3,c=7,d=4

## 📚 Referencias

- Goodrich et al., §8.4 — Tree Traversal Algorithms
- Template Method Pattern: https://refactoring.guru/design-patterns/template-method

---
*Capítulo 8 — Árboles | Estructuras de Datos y Algoritmos — Lic. en Sistemas*  
*Material bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)*